# Detection de Fraude Bancaire — Demonstration

> **Auteur :** Emmanuel TSAGUE — Data Scientist / Data Analyst  
> **Données :** Simulées / Anonymisées — Portfolio pédagogique  
> **GitHub :** https://github.com/TSAGUE25


## 1. Imports

Librairies pour la detection de fraude avec classe rare.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, classification_report
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
print('Imports OK')

## 2. Dataset simule — 100 000 transactions, 0.2% de fraudes

Simulation d'un desequilibre extreme typique de la fraude.


In [ ]:
N_TOTAL = 100000
N_FRAUD = 200
legit = pd.DataFrame({
    'montant': np.abs(np.random.lognormal(3.5, 1.5, N_TOTAL-N_FRAUD)),
    'heure':   np.random.randint(0, 24, N_TOTAL-N_FRAUD),
    'nb_trans_24h': np.random.poisson(3, N_TOTAL-N_FRAUD),
    'dist_km': np.abs(np.random.normal(50, 30, N_TOTAL-N_FRAUD)),
    'fraude': 0,
})
fraude = pd.DataFrame({
    'montant': np.abs(np.random.lognormal(5.5, 2.0, N_FRAUD)),
    'heure':   np.random.choice([0,1,2,3,22,23], N_FRAUD),
    'nb_trans_24h': np.random.poisson(8, N_FRAUD),
    'dist_km': np.abs(np.random.normal(2500, 800, N_FRAUD)),
    'fraude': 1,
})
df = pd.concat([legit, fraude]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Transactions: {len(df):,} | Taux fraude: {df.fraude.mean():.3%}')

## 3. Isolation Forest — detection non supervisee

L'Isolation Forest detecte les anomalies sans labels.


In [ ]:
X = df.drop(columns='fraude')
y = df['fraude']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(n_estimators=100, contamination=0.002, random_state=42)
iso.fit(X_scaled)
y_pred_iso = (iso.predict(X_scaled) == -1).astype(int)
y_score_iso = -iso.score_samples(X_scaled)

pr_auc = average_precision_score(y, y_score_iso)
print(f'Isolation Forest — PR-AUC: {pr_auc:.4f}')
print(classification_report(y, y_pred_iso, target_names=['Legit','Fraude']))